# Laboratorio 1 — Obtención del dato
**Ciencia de Datos · Ingeniería de Ejecución en Informática — Advance UCSC**

| | |
|---|---|
| **Semana** | 1 |
| **Resultado de aprendizaje** | RA1 (foco) · RA3 (transversal) · RA5 (apertura) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Estudiante** | Freddy Antonio Arias Adasme |

---

## Antes de empezar

Sube al entorno los tres archivos que acompañan esta guía:

- `S1_Dataset_Guia_CDD.csv`
- `S1_Base_Ejemplo_CDD.sqlite`
- `S1_Respuesta_API_Ejemplo_CDD.json`

En Google Colab: panel izquierdo → icono de carpeta → *Subir*. En Jupyter local: déjalos en la misma carpeta que este cuaderno.

## Celda 0 · Pregunta de activación

Antes de escribir código, responde aquí para ti (no se entrega):

> **¿Qué dato de tu propio trabajo no usarías hoy para tomar una decisión importante, y por qué?**

No utilizaría un informe de ventas que contenga registros duplicados o información incompleta, porque podría mostrar resultados incorrectos y llevarme a tomar decisiones equivocadas. Antes de utilizar esos datos, revisaría su calidad y verificaría que representen correctamente las operaciones realizadas.

## 1 · Preparación del entorno

Un flujo reproducible parte por declarar qué versiones se usaron. Si alguien ejecuta este cuaderno dentro de un año y obtiene otro resultado, lo primero que va a mirar es esto.

In [10]:
import sys, platform
import pandas as pd
import numpy as np

RUTA = ''          # en Colab déjalo vacío; en local, la carpeta donde están los archivos
SEMILLA = 20261
np.random.seed(SEMILLA)

print('Python    :', sys.version.split()[0], '·', platform.system())
print('pandas    :', pd.__version__)
print('numpy     :', np.__version__)
print('semilla   :', SEMILLA)

Python    : 3.13.16 · Linux
pandas    : 2.2.3
numpy     : 2.1.3
semilla   : 20261


## 2 · Fuente 1 — Archivo plano (CSV)

El primer problema de un archivo plano nunca es el análisis: es **abrirlo bien**. Tres cosas pueden salir mal antes de ver un solo número: la codificación, el separador y los tipos de dato.

Ejecuta la celda siguiente y observa el resultado. Después responde: ¿cuál es el separador de este archivo?

In [11]:
# Lectura deliberadamente ingenua: mira lo que pasa
try:
    crudo = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv')
    print('Forma:', crudo.shape)
    display(crudo.head(3))
except Exception as e:
    print(type(e).__name__)
    print(e)

ParserError
Error tokenizing data. C error: Expected 1 fields in line 39, saw 2



**Lee el error, no lo saltes.** Dice algo como *«Expected 1 fields in line 39, saw 2»*.

Se traduce así: pandas asumió que el separador era la coma, no encontró ninguna en la primera línea y concluyó que el archivo tenía **una sola columna**. Todo iba bien hasta la línea 39, donde apareció una coma —un decimal escrito a la chilena, `12,4`— y esa fila pasó a tener dos campos. Contradicción, y se detiene.

Son dos defectos a la vez: el separador real es `;` y además hay decimales con coma. Resolvamos primero el separador.

In [12]:
aire = pd.read_csv(
    RUTA + 'S1_Dataset_Guia_CDD.csv',
    sep=';',
    encoding='utf-8',
)
print('Forma:', aire.shape)
aire.head()

Forma: (734, 11)


,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado
0,02-20260626,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-06-26,77.1,154.9,9.5,58.3,SI
1,08-20260405,8,Coronel Norte,Coronel,Biobío,2026-04-05,31.6,52.0,12.7,65.4,SI
2,06-20260621,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-06-21,63.5,111.4,6.7,83.0,SI
3,01-20260531,1,Kingston,Concepción,Biobío,2026-05-31,48.1,79.4,14.4,75.1,SI
4,05-20260522,5,Temuco Padre Las Casas,Temuco,Araucanía,2026-05-22,85.0,162.1,11.5,82.9,SI


In [6]:
# Tipos de dato con los que pandas interpretó cada columna
aire.dtypes

,0
id_registro,object
estacion_id,int64
estacion,object
comuna,object
region,object
fecha,object
pm25,float64
pm10,float64
temperatura_c,object
humedad_pct,float64


### ✏️ Actividad 2.1

Mira los tipos con atención. `pm25` y `pm10` quedaron numéricos, pero `temperatura_c` quedó como `object`, es decir, texto. Que una columna numérica quede como texto **siempre** significa que hay algo dentro que no es un número: aquí son los nueve decimales escritos con coma que provocaron el error anterior.

Y existe la trampa inversa: `pm10` es numérico y aun así está contaminado. Lo verás en la actividad 3.2.

**Escribe el código** que muestre los valores no numéricos de `temperatura_c`. Pista: `pd.to_numeric(..., errors='coerce')` convierte a `NaN` lo que no puede convertir.

In [8]:
# Actividad 2.1
# Identificar valores no numéricos en temperatura_c

temperatura_numerica = pd.to_numeric(
    aire['temperatura_c'],
    errors='coerce'
)

valores_no_numericos = aire.loc[
    temperatura_numerica.isna() & aire['temperatura_c'].notna(),
    'temperatura_c'
]

print("Valores no numéricos encontrados:")
print(valores_no_numericos.unique())

print("Cantidad de registros afectados:", len(valores_no_numericos))


Valores no numéricos encontrados:
['7,8' '8,3' '15,0' '10,5' '8,8' '13,3' '16,4' '10,1' '10,4']
Cantidad de registros afectados: 9


## 3 · Perfilado inicial: ¿qué tengo entre manos?

Antes de analizar, hay que **conocer el dato**. Estas cuatro celdas son el mínimo que un analista ejecuta siempre, en este orden.

In [9]:
# a) Dimensiones y nombres de columna
print('Filas   :', aire.shape[0])
print('Columnas:', aire.shape[1])
print()
print(aire.columns.tolist())

Filas   : 734
Columnas: 11

['id_registro', 'estacion_id', 'estacion', 'comuna', 'region', 'fecha', 'pm25', 'pm10', 'temperatura_c', 'humedad_pct', 'validado']


In [10]:
# b) Valores faltantes por columna
faltantes = aire.isna().sum().to_frame('faltantes')
faltantes['porcentaje'] = (faltantes['faltantes'] / len(aire) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,porcentaje
pm25,35,4.77
estacion_id,0,0.00
id_registro,0,0.00
estacion,0,0.00
comuna,0,0.00
region,0,0.00
fecha,0,0.00
pm10,0,0.00
temperatura_c,0,0.00
humedad_pct,0,0.00


In [11]:
# c) Filas duplicadas completas
print('Duplicados exactos:', aire.duplicated().sum())
print('id_registro repetidos:', aire['id_registro'].duplicated().sum())

Duplicados exactos: 14
id_registro repetidos: 14


In [12]:
# d) Cardinalidad de las columnas categóricas
for col in ['estacion', 'comuna', 'region', 'validado']:
    print(f'{col:15s} · {aire[col].nunique():3d} valores distintos')

estacion        ·   8 valores distintos
comuna          ·  16 valores distintos
region          ·   5 valores distintos
validado        ·   1 valores distintos


### ✏️ Actividad 3.1

La columna `comuna` declara más valores distintos de los que deberían existir. **Ejecuta** `aire['comuna'].unique()` y explica en la celda de texto siguiente por qué ocurre.

Ojo: no lo corrijas todavía. Esta semana **detectamos**; en la semana 2 **corregimos** y dejamos registrada la decisión en la bitácora.

In [13]:
aire['comuna'].unique()

array(['Los Ángeles', 'Coronel', 'Osorno', 'Concepción', 'Temuco',
       'Talcahuano', 'Chillán', 'Valdivia', 'TEMUCO  ', 'VALDIVIA  ',
       'LOS ÁNGELES  ', 'TALCAHUANO  ', 'CHILLÁN  ', 'CONCEPCIÓN  ',
       'CORONEL  ', 'OSORNO  '], dtype=object)

La columna comuna presenta 16 valores distintos, pero al revisar los datos se observa que en realidad corresponden a 8 comunas. Esto ocurre porque algunos nombres están escritos con mayúsculas, sin tildes o con espacios adicionales. Por ejemplo, aparecen Los Ángeles y LOS ANGELES, o Concepción y CONCEPCIÓN. Aunque se refieren a las mismas comunas, Python las reconoce como valores diferentes. Por eso, antes de realizar un análisis definitivo, será necesario normalizar los nombres para evitar resultados incorrectos.

### ✏️ Actividad 3.2 — Valores centinela

Un **valor centinela** es un número real que en realidad significa «no hay dato»: `-999`, `0`, `9999`. Si no lo detectas, entra al promedio y lo arruina.

**Escribe el código** que cuente cuántas veces aparece `-999` en `pm10` y calcula el promedio de `pm10` con y sin esos valores. Anota la diferencia.

In [13]:
# Contar valores centinela
cantidad_999 = (aire['pm10'] == -999).sum()

# Calcular promedio incluyendo -999
promedio_con_999 = aire['pm10'].mean()

# Calcular promedio excluyendo -999
promedio_sin_999 = aire.loc[aire['pm10'] != -999, 'pm10'].mean()

# Calcular diferencia
diferencia = promedio_sin_999 - promedio_con_999

print("Cantidad de valores -999:", cantidad_999)
print("Promedio con -999:", round(promedio_con_999, 2))
print("Promedio sin -999:", round(promedio_sin_999, 2))
print("Diferencia:", round(diferencia, 2))



Cantidad de valores -999: 18
Promedio con -999: 77.1
Promedio sin -999: 104.15
Diferencia: 27.05


Se encontraron 18 registros con el valor -999 en la columna pm10. Al calcular el promedio incluyendo estos valores, se obtiene 77,10, mientras que al excluirlos el promedio aumenta a 104,15. La diferencia es de 27,05 unidades. Esto demuestra que los valores centinela afectan los resultados del análisis, por lo que deben identificarse y tratarse adecuadamente antes de utilizar los datos para tomar decisiones.

## 4 · Fuente 2 — Base de datos relacional (SQL)

Buena parte de los datos de una organización no está en archivos, sino en una base de datos. El archivo `S1_Base_Ejemplo_CDD.sqlite` contiene tres tablas: `estacion`, `medicion` y `episodio`.

SQLite viene incluido en Python: no hay que instalar nada.

In [15]:
import sqlite3

con = sqlite3.connect(RUTA + 'S1_Base_Ejemplo_CDD.sqlite')

# ¿Qué tablas hay?
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,estacion
1,medicion
2,sqlite_sequence
3,episodio


In [16]:
# Estructura de una tabla
pd.read_sql_query('PRAGMA table_info(medicion)', con)

,cid,name,type,notnull,dflt_value,pk
0,0,medicion_id,INTEGER,0,None,1
1,1,estacion_id,INTEGER,1,None,0
2,2,fecha,TEXT,1,None,0
3,3,pm25,REAL,0,None,0
4,4,pm10,REAL,0,None,0
5,5,temperatura_c,REAL,0,None,0
6,6,humedad_pct,REAL,0,None,0


In [19]:
# Una consulta con JOIN y agregación: promedio de PM2.5 por comuna
consulta = '''
SELECT  e.comuna,
        e.region,
        COUNT(*)              AS n_mediciones,
        ROUND(AVG(m.pm25), 1) AS pm25_promedio,
        ROUND(MAX(m.pm25), 1) AS pm25_maximo
FROM    medicion m
JOIN    estacion e ON e.estacion_id = m.estacion_id
GROUP BY e.comuna, e.region
ORDER BY pm25_promedio DESC
'''
pd.read_sql_query(consulta, con)

,comuna,region,n_mediciones,pm25_promedio,pm25_maximo
0,Temuco,Araucanía,90,74.2,102.1
1,Chillán,Ñuble,90,65.8,98.5
2,Los Ángeles,Biobío,90,62.2,87.5
3,Osorno,Los Lagos,90,60.3,82.5
4,Valdivia,Los Ríos,90,53.9,78.0
5,Coronel,Biobío,90,49.0,76.5
6,Concepción,Biobío,90,45.8,70.7
7,Talcahuano,Biobío,90,41.3,67.8


### ✏️ Actividad 4.1

**Escribe una consulta SQL** que responda: *¿cuántos episodios de cada categoría (`Alerta`, `Preemergencia`, `Emergencia`) registró cada comuna?*

Necesitas unir `episodio` con `estacion` y agrupar por dos columnas.

In [20]:
# Consultar la estructura de la tabla episodio
pd.read_sql_query("PRAGMA table_info(episodio)", con)

,cid,name,type,notnull,dflt_value,pk
0,0,episodio_id,INTEGER,0,None,1
1,1,estacion_id,INTEGER,1,None,0
2,2,fecha,TEXT,1,None,0
3,3,categoria,TEXT,1,None,0


In [21]:
# Consultar la estructura de la tabla estacion
pd.read_sql_query("PRAGMA table_info(estacion)", con)

,cid,name,type,notnull,dflt_value,pk
0,0,estacion_id,INTEGER,0,None,1
1,1,nombre,TEXT,1,None,0
2,2,comuna,TEXT,1,None,0
3,3,region,TEXT,1,None,0
4,4,latitud,REAL,0,None,0
5,5,longitud,REAL,0,None,0
6,6,fecha_instalacion,TEXT,0,None,0


In [22]:
# Tu consulta aquí
# Actividad 4.1
# Contar episodios de cada categoría por comuna

consulta_2 = '''
SELECT
    e.comuna,
    ep.categoria,
    COUNT(*) AS cantidad_episodios
FROM episodio ep
JOIN estacion e
    ON ep.estacion_id = e.estacion_id
GROUP BY
    e.comuna,
    ep.categoria
ORDER BY
    e.comuna ASC,
    ep.categoria ASC
'''

pd.read_sql_query(consulta_2, con)

,comuna,categoria,cantidad_episodios
0,Chillán,Alerta,5
1,Chillán,Preemergencia,1
2,Los Ángeles,Alerta,6
3,Osorno,Alerta,5
4,Temuco,Alerta,25
5,Temuco,Preemergencia,4


In [ ]:
con.close()

## 5 · Fuente 3 — Servicio web (API) y JSON anidado

Una API devuelve **JSON**, no una tabla. El JSON es jerárquico: dentro de un registro puede haber otro objeto, y dentro de ese, una lista. Convertirlo en una tabla rectangular es parte del trabajo.

Usamos aquí la respuesta guardada en archivo. En un caso real la obtendrías con `requests.get(url).json()`; el archivo existe para que el laboratorio funcione aunque el servicio esté caído.

In [23]:
import json

with open(RUTA + 'S1_Respuesta_API_Ejemplo_CDD.json', encoding='utf-8') as fh:
    respuesta = json.load(fh)

print('Claves de primer nivel:', list(respuesta.keys()))
print()
print('Metadatos:')
for k, v in respuesta['meta'].items():
    print(f'  {k}: {v}')

Claves de primer nivel: ['meta', 'data']

Metadatos:
  servicio: API de ejemplo · Red de monitoreo de calidad del aire (simulada para docencia)
  version: 1.2
  generado: 2026-09-21T10:00:00-03:00
  licencia: CC BY 4.0 (conjunto sintético de uso docente)
  paginacion: {'pagina': 1, 'por_pagina': 100, 'total': 100, 'siguiente': None}


In [24]:
# Un registro cualquiera, para entender la estructura
print(json.dumps(respuesta['data'][0], ensure_ascii=False, indent=2))

{
  "codigo_estacion": "EST-001",
  "ubicacion": {
    "comuna": "Concepción",
    "region": "Biobío",
    "coordenadas": {
      "lat": -36.827,
      "lon": -73.0498
    }
  },
  "fecha": "2026-04-01",
  "mediciones": [
    {
      "parametro": "pm25",
      "valor": 37.6,
      "unidad": "ug/m3"
    },
    {
      "parametro": "pm10",
      "valor": 58.8,
      "unidad": "ug/m3"
    },
    {
      "parametro": "temperatura",
      "valor": 13.1,
      "unidad": "C"
    }
  ],
  "calidad_dato": "preliminar"
}


In [25]:
# json_normalize aplana la jerarquía: fíjate en los nombres de columna que genera
api = pd.json_normalize(respuesta['data'])
api.head(3)

,codigo_estacion,fecha,mediciones,calidad_dato,ubicacion.comuna,ubicacion.region,ubicacion.coordenadas.lat,ubicacion.coordenadas.lon
0,EST-001,2026-04-01,"[{'parametro': 'pm25', 'valor': 37.6, 'unidad'...",preliminar,Concepción,Biobío,-36.827,-73.0498
1,EST-001,2026-04-02,"[{'parametro': 'pm25', 'valor': 25.6, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498
2,EST-001,2026-04-03,"[{'parametro': 'pm25', 'valor': 29.3, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498


In [26]:
# La lista 'mediciones' sigue anidada. record_path la despliega en filas:
api_largo = pd.json_normalize(
    respuesta['data'],
    record_path='mediciones',
    meta=['codigo_estacion', 'fecha', ['ubicacion', 'comuna']],
)
api_largo.head(6)

,parametro,valor,unidad,codigo_estacion,fecha,ubicacion.comuna
0,pm25,37.6,ug/m3,EST-001,2026-04-01,Concepción
1,pm10,58.8,ug/m3,EST-001,2026-04-01,Concepción
2,temperatura,13.1,C,EST-001,2026-04-01,Concepción
3,pm25,25.6,ug/m3,EST-001,2026-04-02,Concepción
4,pm10,63.5,ug/m3,EST-001,2026-04-02,Concepción
5,temperatura,15.6,C,EST-001,2026-04-02,Concepción


### ✏️ Actividad 5.1

El resultado anterior está en **formato largo**: una fila por parámetro medido. Para analizar conviene el **formato ancho**: una fila por estación-fecha y una columna por parámetro.

**Escribe el código** que haga esa transformación con `pivot_table`.

In [27]:
# Actividad 5.1
# Transformar los datos de formato largo a formato ancho

api_ancho = api_largo.pivot_table(
    index=['codigo_estacion', 'fecha', 'ubicacion.comuna'],
    columns='parametro',
    values='valor',
    aggfunc='mean'
).reset_index()

# Eliminar el nombre del índice de columnas
api_ancho.columns.name = None

# Mostrar las primeras 10 filas
print("Datos en formato ancho:")
display(api_ancho.head(10))

# Mostrar dimensiones del resultado
print("Dimensiones:", api_ancho.shape)


Datos en formato ancho:


,codigo_estacion,fecha,ubicacion.comuna,pm10,pm25,temperatura
0,EST-001,2026-04-01,Concepción,58.8,37.6,13.1
1,EST-001,2026-04-02,Concepción,63.5,25.6,15.6
2,EST-001,2026-04-03,Concepción,75.4,29.3,13.3
3,EST-001,2026-04-04,Concepción,55.9,33.0,11.7
4,EST-001,2026-04-05,Concepción,79.1,39.1,12.4
5,EST-001,2026-04-06,Concepción,76.4,42.9,16.3
6,EST-001,2026-04-07,Concepción,69.8,33.7,11.1
7,EST-001,2026-04-08,Concepción,61.3,30.7,11.0
8,EST-001,2026-04-09,Concepción,81.4,29.9,10.8
9,EST-001,2026-04-10,Concepción,75.4,37.0,16.8


Dimensiones: (100, 6)


## 6 · Fuente 4 — Extracción web

Cuando el dato solo existe publicado en una página, hay que extraerlo. `pandas.read_html` resuelve el caso más común: una tabla HTML.

**Antes de extraer, tres preguntas que no son técnicas:** ¿los términos de uso del sitio lo permiten?, ¿estás generando una carga razonable?, ¿el dato incluye personas? Si alguna respuesta te incomoda, no extraigas.

In [28]:
# Requiere conexión a internet. Si no la tienes, salta esta celda.
URL = 'https://es.wikipedia.org/wiki/Anexo:Comunas_de_Chile'   # [DOCENTE: confirmar o reemplazar]

try:
    tablas = pd.read_html(URL)
    print('Tablas encontradas:', len(tablas))
    web = tablas[0]
    display(web.head())
except Exception as e:
    print('No se pudo extraer:', type(e).__name__, e)
    print('Continúa con el resto del laboratorio.')

No se pudo extraer: HTTPError HTTP Error 403: Forbidden
Continúa con el resto del laboratorio.


### ✏️ Actividad 6.1

Responde en la celda de texto siguiente: **¿qué diferencia hay, para tu proyecto, entre un dato descargado de un portal oficial y uno extraído de una página web?** Piensa en trazabilidad y en qué pasa si mañana la página cambia.

La principal diferencia es que los datos descargados desde un portal oficial generalmente tienen una fuente identificada, documentación y una estructura más estable, lo que facilita comprobar su origen y utilizarlos en un análisis. En cambio, cuando se extraen datos directamente de una página web, puede ser más difícil asegurar su trazabilidad, ya que la información podría cambiar sin previo aviso. Además, si la página modifica su estructura, el código utilizado para extraer los datos podría dejar de funcionar. Por eso, para mi proyecto preferiría utilizar datos de fuentes oficiales, siempre que estén disponibles, y registrar su origen y fecha de descarga.



## 7 · Volumen: cuando el conjunto no cabe

No todo conjunto se lee entero. Cuando el archivo supera la memoria disponible hay tres salidas: **leer solo las columnas necesarias**, **leer por bloques** o **trabajar con una muestra**.

Una muestra bien construida responde muchas preguntas con una fracción del costo. Una muestra mal construida responde mal todas.

In [29]:
# Leer solo algunas columnas: menos memoria desde el inicio
liviano = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';',
                      usecols=['estacion', 'fecha', 'pm25'])
print('Memoria completo :', aire.memory_usage(deep=True).sum() / 1024, 'KB')
print('Memoria liviano  :', liviano.memory_usage(deep=True).sum() / 1024, 'KB')

Memoria completo : 335.0009765625 KB
Memoria liviano  : 94.8486328125 KB


In [30]:
# Lectura por bloques: se procesa de a trozos y nunca se carga todo
total, filas = 0, 0
for bloque in pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';', chunksize=100):
    valores = pd.to_numeric(bloque['pm25'], errors='coerce').dropna()
    total += valores.sum()
    filas += len(valores)
print(f'Promedio de pm25 calculado por bloques: {total / filas:.2f}')

Promedio de pm25 calculado por bloques: 62.04


In [31]:
# Muestreo aleatorio simple vs. estratificado por estación
simple = aire.sample(n=120, random_state=SEMILLA)
estratificado = (aire.groupby('estacion', group_keys=False)
                     .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))

print('Distribución en la muestra SIMPLE:')
print(simple['estacion'].value_counts().to_string())
print()
print('Distribución en la muestra ESTRATIFICADA:')
print(estratificado['estacion'].value_counts().to_string())

Distribución en la muestra SIMPLE:
estacion
Kingston                  18
Osorno Entre Lagos        18
Coyhanco                  17
Los Ángeles Oriente       17
Coronel Norte             16
Talcahuano                13
Temuco Padre Las Casas    13
Valdivia Centro            8

Distribución en la muestra ESTRATIFICADA:
estacion
Coronel Norte             15
Coyhanco                  15
Kingston                  15
Los Ángeles Oriente       15
Osorno Entre Lagos        15
Talcahuano                15
Temuco Padre Las Casas    15
Valdivia Centro           15


/tmp/ipykernel_2824/748870216.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))


### ✏️ Actividad 7.1

Mira las dos distribuciones anteriores. **¿En qué situación la muestra simple te llevaría a una conclusión equivocada?** Escríbelo abajo con un ejemplo concreto de tu propio conjunto de datos.

Una muestra aleatoria simple podría llevarme a una conclusión equivocada si quiero comparar los niveles de contaminación entre distintas estaciones. En el ejercicio, por ejemplo, la estación Kingston quedó representada con 18 registros, mientras que Valdivia Centro tuvo solamente 8. Si utilizara esa muestra para comparar los niveles de contaminación, algunas estaciones tendrían menos información disponible y sus resultados podrían ser menos precisos. En cambio, el muestreo estratificado seleccionó 15 registros de cada estación, permitiendo realizar una comparación más equilibrada y reduciendo el riesgo de que alguna estación quede poco representada.



## 8 · Tu conjunto de datos: la elección del proyecto

Abre `S1_Catalogo_Datasets_Sugeridos_CDD.xlsx`, revisa **al menos tres** alternativas y elige una. Registra tu elección en la hoja «Mi elección» del mismo archivo.

Luego descárgalo, cárgalo aquí abajo y ejecuta sobre él el mismo perfilado de la sección 3.

In [32]:
# Cargar el conjunto de datos Online Retail

mi_dataset = pd.read_excel(
    RUTA + 'Online Retail.xlsx',
    engine='openpyxl'
)

print("Filas:", mi_dataset.shape[0])
print("Columnas:", mi_dataset.shape[1])

display(mi_dataset.head())


Filas: 541909
Columnas: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [33]:
# PERFILADO INICIAL - ONLINE RETAIL

# 1. Dimensiones
print("=== DIMENSIONES ===")
print("Filas:", mi_dataset.shape[0])
print("Columnas:", mi_dataset.shape[1])
print("Nombres de columnas:", mi_dataset.columns.tolist())

# 2. Valores faltantes
print("\n=== VALORES FALTANTES ===")

faltantes = mi_dataset.isna().sum().to_frame('faltantes')
faltantes['porcentaje'] = (
    faltantes['faltantes'] / len(mi_dataset) * 100
).round(2)

display(faltantes.sort_values('faltantes', ascending=False))

# 3. Registros duplicados
print("\n=== DUPLICADOS ===")
print("Duplicados exactos:", mi_dataset.duplicated().sum())

# 4. Cardinalidad
print("\n=== CARDINALIDAD ===")

for columna in mi_dataset.columns:
    print(
        f"{columna}: "
        f"{mi_dataset[columna].nunique()} valores distintos"
    )


=== DIMENSIONES ===
Filas: 541909
Columnas: 8
Nombres de columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

=== VALORES FALTANTES ===


,faltantes,porcentaje
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00



=== DUPLICADOS ===
Duplicados exactos: 5268

=== CARDINALIDAD ===
InvoiceNo: 25900 valores distintos
StockCode: 4070 valores distintos
Description: 4223 valores distintos
Quantity: 722 valores distintos
InvoiceDate: 23260 valores distintos
UnitPrice: 1630 valores distintos
CustomerID: 4372 valores distintos
Country: 38 valores distintos


### ✏️ Actividad 8.1 — Pregunta analítica y unidad de análisis

Completa estas dos líneas. Son la base de todo el curso y las vas a necesitar en la Entrega 1.

- **Mi pregunta analítica es:**
¿Cuáles son los productos que generan mayores ingresos en el comercio electrónico y cómo varían sus ventas a lo largo del tiempo?

- **Una fila de mi conjunto representa:**
Cada fila representa una línea de una transacción comercial, correspondiente a un producto incluido en una factura. Contiene información sobre el producto, la cantidad vendida, el precio unitario, la fecha, el cliente y el país de la transacción.

Criterio para saber si la pregunta está bien formulada: si al leerla puedes nombrar qué columnas necesitas para responderla, está bien. Si no, es demasiado vaga.

## 9 · Riesgo ético y legal

Antes de usar tu conjunto, completa la ficha `S1_Ficha_Riesgo_Etico_CDD.docx`. No es un trámite: si tu conjunto contiene datos de personas, lo que decidas ahí condiciona todo lo que puedes hacer en las semanas siguientes.

Esa ficha es el insumo de tu intervención en el **foro de debate** de esta semana y se cierra en la Sección C de la Entrega Final.

## 10 · Repositorio y control de versiones

Tu trabajo tiene que poder reconstruirse. Sube este cuaderno a un repositorio y deja el enlace aquí.

Desde Colab: *Archivo → Guardar una copia en GitHub*. Desde local:

In [ ]:
# Ejecuta esto en una terminal, no en el cuaderno:
# git init
# git add S1_Notebook_Base_CDD.ipynb
# git commit -m 'Laboratorio 1: obtención del dato'
# git remote add origin https://github.com/USUARIO/proyecto-ciencia-datos.git
# git push -u origin main


**Enlace a mi repositorio:**

https://github.com/fariasad/UCSC-Ciencia-de-Datos

---

## Cierre del laboratorio

Al terminar deberías tener:

- [x] El entorno funcionando y las versiones declaradas
- [x] Datos leídos desde las cuatro fuentes
- [x] El perfilado inicial del conjunto guía, con sus defectos detectados (no corregidos)
- [x] Tu conjunto de datos elegido y registrado en el catálogo
- [x] Tu pregunta analítica y tu unidad de análisis escritas
- [x] La ficha de riesgo ético-legal completada
- [x] El repositorio creado y este cuaderno versionado

Lo que quede pendiente, llévalo a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 1_